# Three different ways we use in this hands on project to talk to a database: raw SQL → ORM → ORM inside a web API

| | Part 1 – `sqlite3` | Part 2 – SQLAlchemy ORM | Part 3 – Flask + Flask-SQLAlchemy + Marshmallow |
|---|---|---|---|
| Table is defined as | an SQL `CREATE TABLE` string | a Python class (`Base`) | a Python class (`db.Model`) |
| You write | SQL text | Python objects | Python objects + HTTP routes |
| Read results come back as | tuples `(1, 'Albert Einstein', ...)` | `Winner` objects | `Winner` objects → text (`/winners/`) or JSON via Marshmallow (`/api/winners/`) |
| Connection handling | your cursor + `commit()` | engine + session you create | Flask manages `db.session` |
| Who can use the data | this script only | this script only | any client over HTTP (browser, Postman, JS, `requests`) |
| Change database type (e.g. PostgreSQL) | rewrite the code | change one URL | change one config value |


In [1]:
# %pip install --upgrade pip

In [2]:
# Framework, Libraries, Extension
# %pip install -q flask flask-sqlalchemy flask-marshmallow marshmallow-sqlalchemy pandas

# or 
# %pip install -q flask
# %pip install -q flask-sqlalchemy
# %pip install -q flask-marshmallow
# %pip install -q marshmallow-sqlalchemy
# %pip install -q pandas


In [3]:
#one 'data' folder, one database file for each part of the project
import os

BASE_DIR = os.getcwd()                         # folder this notebook is running in
DATA_FOLDER = os.path.join(BASE_DIR, "data")
os.makedirs(DATA_FOLDER, exist_ok=True)        # create 'data/' if it doesn't exist

# Absolute paths, so the files always land in the same place
PART1_DB = os.path.join(DATA_FOLDER, "part1_sqlite3.db")
PART2_DB = os.path.join(DATA_FOLDER, "part2_sqlalchemy.db")
PART3_DB = os.path.join(DATA_FOLDER, "part3_flask.db")

# The same seed data is used in every part, so results are comparable
NOBLE_WINNERS = [
    ("Albert Einstein", "Physics",    "Germany"),
    ("Marie Curie",     "Chemistry",  "Poland"),
    ("Bob Dylan",       "Literature", "USA"),
]

## Part 1 – Raw SQL with `sqlite3`

Write every SQL statement as a string, send it through a **cursor**,
and get rows back as plain **tuples** (no column names attached). 
Nothing is saved to disk until a call to `commit()` is made.

In [4]:
import sqlite3

# Open (or create) the database file. 'conn' is the connection, 'cursor' sends SQL through it.
conn = sqlite3.connect(PART1_DB)
cursor = conn.cursor()

# Define the table in SQL. Every type and constraint is written by hand.
cursor.execute('''
CREATE TABLE IF NOT EXISTS winners (
    id       INTEGER PRIMARY KEY,   
    name     TEXT NOT NULL,
    category TEXT,
    country  TEXT
)
''')
conn.commit()
print("Part 1 database path:", os.path.relpath(PART1_DB))

Part 1 database path: data/part1_sqlite3.db


In [5]:
#NOBLE_WINNER: will insert the starting rows only if the table is empty

count = cursor.execute("SELECT COUNT(*) FROM winners").fetchone()[0]
if count == 0:                                      # '?' placeholders = parameterized query: sqlite3 inserts the values safely
                                                    # (protects against SQL injection). executemany runs the INSERT once per tuple.
    cursor.executemany(
        "INSERT INTO winners (name, category, country) VALUES (?, ?, ?)",
        NOBLE_WINNERS,
    )
    conn.commit()
    print("Seed data added")
else:
    print(f"Table already has {count} rows")

Table already has 3 rows


In [6]:
NOBLE_WINNERS

[('Albert Einstein', 'Physics', 'Germany'),
 ('Marie Curie', 'Chemistry', 'Poland'),
 ('Bob Dylan', 'Literature', 'USA')]

In [7]:
#READ: results are tuples
rows = cursor.execute("SELECT * FROM winners").fetchall()
print("Raw result in database:", rows)      

# # Tuples have no column names; you must remember the order yourself,

# # or read the names from cursor.description:
columns = [col[0] for col in cursor.description]
print("Columns:  ", columns)


def show_part1(label):
    print(f"\n {label} ")             # Printing the whole table so we can watch it change in databse
    for row in cursor.execute("SELECT * FROM winners ORDER BY id"):
        print(row)

show_part1("Part1 table1")

Raw result in database: [(1, 'Albert Einstein', 'Physics', 'Germany'), (2, 'Marie Curie', 'Chemistry', 'Poland'), (3, 'Bob Dylan', 'Literature', 'USA')]
Columns:   ['id', 'name', 'category', 'country']

 Part1 table1 
(1, 'Albert Einstein', 'Physics', 'Germany')
(2, 'Marie Curie', 'Chemistry', 'Poland')
(3, 'Bob Dylan', 'Literature', 'USA')


In [8]:
#CREATE / UPDATE / DELETE, all as SQL strings

# CREATE the new data entry
cursor.execute(
    "INSERT INTO winners (name, category, country) VALUES (?, ?, ?)",
    ("C.V. Raman", "Physics", "India"),
)
conn.commit()
new_id = cursor.lastrowid                 # id SQLite just assigned
show_part1(f"After INSERT (new id = {new_id})")

# UPDATE the data entry in the database
cursor.execute("UPDATE winners SET country = ? WHERE id = ?", ("British India", new_id))
conn.commit()
show_part1("After UPDATE")

# DELETE
cursor.execute("DELETE FROM winners WHERE id = ?", (new_id,))           #delete by the id we just got, not by hard-coded ids like (1, 2)
conn.commit()
show_part1("After DELETE")


 After INSERT (new id = 4) 
(1, 'Albert Einstein', 'Physics', 'Germany')
(2, 'Marie Curie', 'Chemistry', 'Poland')
(3, 'Bob Dylan', 'Literature', 'USA')
(4, 'C.V. Raman', 'Physics', 'India')

 After UPDATE 
(1, 'Albert Einstein', 'Physics', 'Germany')
(2, 'Marie Curie', 'Chemistry', 'Poland')
(3, 'Bob Dylan', 'Literature', 'USA')
(4, 'C.V. Raman', 'Physics', 'British India')

 After DELETE 
(1, 'Albert Einstein', 'Physics', 'Germany')
(2, 'Marie Curie', 'Chemistry', 'Poland')
(3, 'Bob Dylan', 'Literature', 'USA')


In [9]:
conn.close()   # always close the connection when finished

## Part 2 – SQLAlchemy ORM

An **ORM** (Object-Relational Mapper) lets you describe the table as a **Python class** and
work with **objects** instead of SQL strings. SQLAlchemy writes the SQL for you.

* **Engine** – manages the connection to the database file.
* **Base** (`declarative_base()`) – any class inheriting from it becomes a table.
* **Session** – a workspace: stage changes with `add()`, save them with `commit()`.

| Part 1 (SQL) | Part 2 (ORM) |
|---|---|
| `CREATE TABLE winners (...)` | `class Winner(Base)` + `create_all()` |
| `INSERT INTO winners ...` | `session.add(Winner(...))` |
| `SELECT * FROM winners` | `session.query(Winner).all()` |
| `UPDATE winners SET country=?` | `winner.country = "..."` |
| `DELETE FROM winners WHERE id=?` | `session.delete(winner)` |

In [10]:
# from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import create_engine, Column, String, Integer

# echo=False keeps the output readable; set it to True to print every SQL statement SQLAlchemy generates
engine = create_engine(f"sqlite:///{PART2_DB}", echo=False)
Base = declarative_base()


# MODEL: this class IS the table definition
class Winner(Base):
    __tablename__ = "winners"

    id = Column(Integer, primary_key=True)     # auto-generated
    name = Column(String, nullable=False)      # same rule as 'TEXT NOT NULL' in Part1
    category = Column(String)
    country = Column(String)

    # Without __repr__, print() shows '<Winner object at 0x...>' instead of the data
    def __repr__(self):
        return f"<Winner id={self.id} name={self.name!r} category={self.category!r} country={self.country!r}>"

Base.metadata.create_all(engine)     # CREATE TABLE for every model that doesn't exist yet
Session = sessionmaker(bind=engine)  # a factory that makes sessions
print("Part 2 database:", os.path.relpath(PART2_DB))

Part 2 database: data/part2_sqlalchemy.db


In [11]:
# 'with Session() as session' closes the session automatically at the end.

with Session() as session:
    if session.query(Winner).count() == 0:
        session.add_all([Winner(name=n, category=c, country=co) for n, c, co in NOBLE_WINNERS])
        session.commit()
        print("Seed data added")
    else:
        print("Table already has data")

Table already has data


In [12]:
print(session.query(Winner).all())

[<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>, <Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>, <Winner id=3 name='Bob Dylan' category='Literature' country='USA'>]


In [13]:
#READ: results are Winner objects, fields accessed by name
def show_part2(label):
    print(f"\n {label} ")
    with Session() as session:
        for w in session.query(Winner).order_by(Winner.id).all():
            print(w)                                

show_part2("Part2 table")

# Filtering without writing SQL command:  SELECT * FROM winners WHERE category = 'Physics'
with Session() as session:
    physicists = session.query(Winner).filter_by(category="Physics").all()
    print("\nPhysics winners:", [w.name for w in physicists])


 Part2 table 
<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'>

Physics winners: ['Albert Einstein']


In [14]:
#CREATE / UPDATE / DELETE with objects

with Session() as session:
    # CREATE: build an object, stage it, commit -> SQLAlchemy runs the INSERT
    raman = Winner(name="C.V. Raman", category="Physics", country="India")
    session.add(raman)
    session.commit()
    new_id = raman.id                     # id is filled in after the commit
    show_part2(f"After INSERT (new id = {new_id})")

    # UPDATE: just change the attribute, then commit -> SQLAlchemy runs the UPDATE
    raman.country = "Bharat India"
    session.commit()
    show_part2("After UPDATE")

    # DELETE
    session.delete(raman)
    session.commit()
    show_part2("After DELETE")


 After INSERT (new id = 4) 
<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'>
<Winner id=4 name='C.V. Raman' category='Physics' country='India'>

 After UPDATE 
<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'>
<Winner id=4 name='C.V. Raman' category='Physics' country='Bharat India'>

 After DELETE 
<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'>


In [15]:
#OPTIONAL: see the SQL the ORM writes for you
engine.echo = True                        # turn SQL logging on
with Session() as session:
    session.query(Winner).filter_by(country="Poland").all()
engine.echo = False                     # and off again

2026-10-14 16:42:41,691 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-14 16:42:41,693 INFO sqlalchemy.engine.Engine SELECT winners.id AS winners_id, winners.name AS winners_name, winners.category AS winners_category, winners.country AS winners_country 
FROM winners 
WHERE winners.country = ?
2026-10-14 16:42:41,694 INFO sqlalchemy.engine.Engine [generated in 0.00103s] ('Poland',)
2026-10-14 16:42:41,696 INFO sqlalchemy.engine.Engine ROLLBACK


## Part 3 – Flask + Flask-SQLAlchemy, then Marshmallow (ORM inside a web API)

Same ORM idea as Part 2, with Flask on top. Part 3 is built in two steps on **one app and one database**:

* **3a – Flask without JSON** (`/winners/...`): routes read **form fields** and reply with the objects'
  `__repr__` **text**. No serialization: the reply is readable for people, but hard for programs to use.
* **3b – Marshmallow** (`/api/winners/...`): the same five routes, but data goes **out** through
  `dump()` + `jsonify` (serialization) and comes **in** through `get_json()` + `load()` (deserialization).

| HTTP request | Text version (3a) | JSON + Marshmallow version (3b) | Database operation |
|---|---|---|---|
| GET all | `GET /winners/` | `GET /api/winners/` | read all |
| GET one | `GET /winners/<id>` | `GET /api/winners/<id>` | read one |
| Create | `POST /winners/` | `POST /api/winners/` | create |
| Update | `PUT /winners/<id>` | `PUT /api/winners/<id>` | update |
| Delete | `DELETE /winners/<id>` | `DELETE /api/winners/<id>` | delete |

**Cell-order rule for Part 3.** Flask does not allow new routes once the app has handled a request.
So **every cell that defines routes comes before every cell that sends a request** (the tests in 3c).
If you change a route, re-run from the `app = Flask(__name__)` cell downwards, or use Restart + Run All.
Never re-run a route cell on its own: it registers the same routes twice and Flask raises an error.

### Part 3a – Flask app, model and text routes (no JSON)

In [16]:
from flask import Flask, request
from flask_sqlalchemy import SQLAlchemy
from flask_marshmallow import Marshmallow
from flask import Response
import pandas as pd


#FLASK APP + CONFIG
app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{PART3_DB}"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

#EXTENSIONS 
db = SQLAlchemy(app)    # gives db.Model, db.session, db.engine, db.Column, db.create_all()
ma = Marshmallow(app)   # gives ma.SQLAlchemyAutoSchema


In [17]:
# Model (inherits from db.Model instead of Part 2's Base)

class Winner(db.Model):
    __tablename__ = "winners"

    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String, nullable=False)
    category = db.Column(db.String)
    country = db.Column(db.String)

    # This text is what every route sends back, since there is no JSON
    def __repr__(self):
        return f"<Winner id={self.id} name={self.name!r} category={self.category!r} country={self.country!r}>"


REQUIRED_FIELDS = ("name", "category", "country")


# Helper: every route answers with plain text.
# mimetype="text/plain" stops the browser from treating <Winner ...> as an HTML tag and hiding it.

def text_response(body, status=200):
    return Response(body, status=status, mimetype="text/plain")


In [18]:
# Routes

@app.route("/")
def home():
    return text_response("Winners API")


@app.route("/winners/", methods=["GET"])
def get_winners():
    all_winners = Winner.query.order_by(Winner.id).all()      # SELECT * FROM winners
    # One object per line, using __repr__
    return text_response("\n".join(repr(w) for w in all_winners))


@app.route("/winners/<int:id>", methods=["GET"])
def get_winner(id):
    winner = db.session.get(Winner, id)                       # SELECT ... WHERE id = ?
    if winner is None:
        return text_response("Winner not found", 404)
    return text_response(repr(winner))


@app.route("/winners/", methods=["POST"])
def create_winner():
    # Input WITHOUT JSON: form fields, the way an HTML form sends data
    data = request.form

    missing = [f for f in REQUIRED_FIELDS if not data.get(f)]
    if missing:
        return text_response(f"Missing fields: {', '.join(missing)}", 400)

    new_winner = Winner(name=data["name"], category=data["category"], country=data["country"])
    db.session.add(new_winner)
    db.session.commit()                                       # INSERT
    return text_response(repr(new_winner), 201)               # 201 = Created


@app.route("/winners/<int:id>", methods=["PUT"])
def update_winner(id):
    winner = db.session.get(Winner, id)
    if winner is None:
        return text_response("Winner not found", 404)

    data = request.form
    if not data:
        return text_response("Send at least one form field to update", 400)

    # Only fields that were sent are changed; the rest keep their old values
    winner.name = data.get("name", winner.name)
    winner.category = data.get("category", winner.category)
    winner.country = data.get("country", winner.country)
    db.session.commit()                                       # UPDATE
    return text_response(repr(winner))


@app.route("/winners/<int:id>", methods=["DELETE"])
def delete_winner(id):
    winner = db.session.get(Winner, id)
    if winner is None:
        return text_response("Winner not found", 404)
    db.session.delete(winner)
    db.session.commit()                                       # DELETE
    return text_response(f"Deleted: {winner!r}")



In [19]:
# Create table + seed (outside a request, so an app context is needed)

with app.app_context():
    db.create_all()
    if Winner.query.count() == 0:
        db.session.add_all([Winner(name=n, category=c, country=co) for n, c, co in NOBLE_WINNERS])
        db.session.commit()
        print("Seed data added")
    print("Part3 database:", os.path.relpath(PART3_DB))


Part3 database: data/part3_flask.db


In [20]:
# Helper: read the 'winners' table DIRECTLY with SQL (not through the API),
# so after each request we can see what is actually stored in the database
def show_part3(label):
    with app.app_context():
        df = pd.read_sql("SELECT * FROM winners ORDER BY id", db.engine)
    print(f"\n {label} ")
    print(df.to_string(index=False))

show_part3("Part 3 table")


 Part 3 table 
 id            name   category country
  1 Albert Einstein    Physics Germany
  2     Marie Curie  Chemistry  Poland
  3       Bob Dylan Literature     USA


## Part 3b – Serialization and deserialization with Marshmallow

* **Serialization**: Winner object → dictionary (`dump()`) → JSON text (`dumps()`, or `jsonify` in a route).
* **Deserialization**: JSON text → dictionary (`json.loads()` / `request.get_json()`) → checked Winner object (`load()`).

The schema below is the translator between the `Winner` model and plain data. The next two cells call it
directly (outside any route) to show each step; after that, the JSON routes use it for real requests.
An app context is needed in these demo cells because they use the database outside a request.

In [21]:
# Marshmallow schema: the translator between Winner objects and plain data

from marshmallow import ValidationError

class WinnerSchema(ma.SQLAlchemyAutoSchema):
    class Meta:
        model = Winner            # read the columns from the model
        load_instance = True      # load() gives back a Winner object, not just a dictionary
        dump_only = ("id",)       

    # The model allows empty category/country, but the API should require them
    category = ma.auto_field(required=True)
    country = ma.auto_field(required=True)

winner_schema = WinnerSchema()                  # for one object
multiwinners_schema = WinnerSchema(many=True)   # for a list of objects

In [22]:
 # SERIALIZATION: Winner object -> dictionary -> JSON text

with app.app_context():
    all_winners = Winner.query.all()
    print("From the database:", type(all_winners[0]))
    print(all_winners)

    result = multiwinners_schema.dump(all_winners)          # object -> dict
    print("\nAfter dump():", type(result), "of", type(result[0]))
    print(result)

    result_json = multiwinners_schema.dumps(all_winners)    # object -> JSON text
    print("\nAfter dumps():", type(result_json))
    print(result_json)

    single_winner = Winner.query.first()
    print("\nSingle winner, dump(): ", winner_schema.dump(single_winner))
    print("Single winner, dumps():", winner_schema.dumps(single_winner))


From the database: <class '__main__.Winner'>
[<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>, <Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>, <Winner id=3 name='Bob Dylan' category='Literature' country='USA'>]

After dump(): <class 'list'> of <class 'dict'>
[{'category': 'Physics', 'country': 'Germany', 'id': 1, 'name': 'Albert Einstein'}, {'category': 'Chemistry', 'country': 'Poland', 'id': 2, 'name': 'Marie Curie'}, {'category': 'Literature', 'country': 'USA', 'id': 3, 'name': 'Bob Dylan'}]

After dumps(): <class 'str'>
[{"category": "Physics", "country": "Germany", "id": 1, "name": "Albert Einstein"}, {"category": "Chemistry", "country": "Poland", "id": 2, "name": "Marie Curie"}, {"category": "Literature", "country": "USA", "id": 3, "name": "Bob Dylan"}]

Single winner, dump():  {'category': 'Physics', 'country': 'Germany', 'id': 1, 'name': 'Albert Einstein'}
Single winner, dumps(): {"category": "Physics", "country": "Germany", "id"

In [23]:
# DESERIALIZATION: JSON text -> dictionary -> checked Winner object

import json

incoming = '{"name": "C.V. Raman", "category": "Physics", "country": "India"}'   # what a client sends
print("Received:", type(incoming), incoming)

with app.app_context():
    data = json.loads(incoming)                 # JSON text -> dict (this is what request.get_json() does)
    print("\nAfter json.loads():", type(data), data)

    new_winner = winner_schema.load(data)       # dict -> checked Winner object
    print("\nAfter load():", type(new_winner), new_winner)
    print("id is", new_winner.id, "because it is not saved yet (no add/commit)")

    same_winner = winner_schema.loads(incoming) # JSON text -> Winner object, in one step
    print("\nAfter loads():", same_winner)

    # load() also CHECKS the data before building anything
    bad_inputs = [
        {"name": "Only a name"},                                           # missing fields
        {"name": 123, "category": "Physics", "country": "India"},          # wrong type
        {"id": 50, "name": "X", "category": "Y", "country": "Z"},          # trying to set id
    ]
    print()
    for bad in bad_inputs:
        try:
            winner_schema.load(bad)
        except ValidationError as err:
            print("Rejected", bad, "->", err.messages)


Received: <class 'str'> {"name": "C.V. Raman", "category": "Physics", "country": "India"}

After json.loads(): <class 'dict'> {'name': 'C.V. Raman', 'category': 'Physics', 'country': 'India'}

After load(): <class '__main__.Winner'> <Winner id=None name='C.V. Raman' category='Physics' country='India'>
id is None because it is not saved yet (no add/commit)

After loads(): <Winner id=None name='C.V. Raman' category='Physics' country='India'>

Rejected {'name': 'Only a name'} -> {'category': ['Missing data for required field.'], 'country': ['Missing data for required field.']}
Rejected {'name': 123, 'category': 'Physics', 'country': 'India'} -> {'name': ['Not a valid string.']}
Rejected {'id': 50, 'name': 'X', 'category': 'Y', 'country': 'Z'} -> {'id': ['Unknown field.']}


### JSON routes: the same API, now with Marshmallow

Same five operations as the text routes, at `/api/winners/...`, on the same app and database.
* **Out** (every method): `schema.dump()` → `jsonify` (serialization)
* **In** (POST, PUT): `request.get_json()` → `schema.load()` (deserialization + validation)
* GET and DELETE have no request body, so they only serialize the reply.

In [24]:
# JSON ROUTES: JSON + Marshmallow versions of the same routes, under /api/ (same app, same database)
from flask import jsonify

@app.route("/api/winners/", methods=["GET"])
def api_get_winners():
    all_winners = Winner.query.order_by(Winner.id).all()
    return jsonify(multiwinners_schema.dump(all_winners)), 200      # serialize: objects -> dicts -> JSON


@app.route("/api/winners/<int:id>", methods=["GET"])
def api_get_winner(id):
    winner = db.session.get(Winner, id)
    if winner is None:
        return jsonify({"error": "Winner not found"}), 404
    return jsonify(winner_schema.dump(winner)), 200


@app.route("/api/winners/", methods=["POST"])
def api_create_winner():
    data = request.get_json(silent=True)                 # deserialize step 1: JSON text -> dict
    if data is None:
        return jsonify({"error": "Request body must be JSON"}), 400
    try:
        new_winner = winner_schema.load(data)            # deserialize step 2: dict -> checked Winner
    except ValidationError as err:
        return jsonify({"errors": err.messages}), 400
    db.session.add(new_winner)
    db.session.commit()                                  # INSERT
    return jsonify(winner_schema.dump(new_winner)), 201  # serialize the saved winner (now with an id)


@app.route("/api/winners/<int:id>", methods=["PUT"])
def api_update_winner(id):
    winner = db.session.get(Winner, id)
    if winner is None:
        return jsonify({"error": "Winner not found"}), 404
    data = request.get_json(silent=True)
    if data is None:
        return jsonify({"error": "Request body must be JSON"}), 400
    try:
        # instance=winner: update this existing object; partial=True: only the sent fields are needed
        winner = winner_schema.load(data, instance=winner, partial=True)
    except ValidationError as err:
        return jsonify({"errors": err.messages}), 400
    db.session.commit()                                  # UPDATE
    return jsonify(winner_schema.dump(winner)), 200


@app.route("/api/winners/search", methods=["GET"])
def search_winners():
    country = request.args.get("country")
    category = request.args.get("category")

    query = Winner.query

    if country:
        query = query.filter_by(country=country)

    if category:
        query = query.filter_by(category=category)

    winners = query.all()

    print("HI method winners search")

    return jsonify(multiwinners_schema.dump(winners))

@app.route("/api/winners/<int:id>", methods=["DELETE"])
def api_delete_winner(id):
    winner = db.session.get(Winner, id)
    if winner is None:
        return jsonify({"error": "Winner not found"}), 404
    deleted = winner_schema.dump(winner)                 # serialize before it is gone
    db.session.delete(winner)
    db.session.commit()                                  # DELETE
    return jsonify({"message": "Winner deleted", "deleted": deleted}), 200




In [25]:
# client = app.test_client()

# r = client.get("/api/winners/search?country=India")

# print(r.status_code)
# print(r.get_json())

### Live view: request log + web page

`@app.after_request` runs after **every** request, whether it comes from the browser or from the notebook's
test client, and records it in a log. The `/ui` page shows the database table and that log, refreshing
every second, and has buttons to send GET / POST / PUT / DELETE to either version of the API.
This cell defines routes, so it comes before the test cells.

In [26]:
# Live view: a request log + a web page (/ui) that shows the database changing
# This cell defines routes, so it must run BEFORE any test cell.
import json
from collections import deque
from datetime import datetime

REQUEST_LOG = deque(maxlen=30)      # the last 30 requests, newest first


# after_request runs after EVERY request (from the browser or from the notebook's test client)
@app.after_request
def log_request(response):
    # Skip the page's own automatic refreshes, otherwise the log fills up every second
    if request.headers.get("X-Live-View") == "poll" or request.path in ("/ui", "/api/log", "/favicon.ico"):
        return response

    sent = request.get_json(silent=True) if request.is_json else request.form.to_dict()
    REQUEST_LOG.appendleft({
        "time": datetime.now().strftime("%H:%M:%S"),
        "source": "browser" if "Mozilla" in request.headers.get("User-Agent", "") else "notebook",
        "method": request.method,
        "path": request.path,
        "status": response.status_code,
        "sent": json.dumps(sent) if sent else "",
        "reply": response.get_data(as_text=True)[:300],
    })
    return response


@app.route("/api/log")
def api_log():
    return jsonify(list(REQUEST_LOG))


UI_PAGE = r"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Winners API: live view</title>
<style>
  body { font-family: system-ui, sans-serif; margin: 20px; background: #fafafa; color: #222; }
  h1 { font-size: 20px; margin: 0 0 4px; }
  .sub { color: #666; margin: 0 0 16px; font-size: 14px; }
  .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
  .card { background: #fff; border: 1px solid #ddd; border-radius: 8px; padding: 14px; }
  h2 { font-size: 15px; margin: 0 0 10px; }
  label { font-size: 13px; color: #555; }
  input, select { padding: 6px; margin: 3px 0; width: 100%; box-sizing: border-box; }
  .row { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; }
  button { padding: 7px 10px; margin: 6px 4px 0 0; cursor: pointer; border: 1px solid #bbb; border-radius: 6px; background: #f0f0f0; }
  table { border-collapse: collapse; width: 100%; font-size: 14px; }
  td, th { border-bottom: 1px solid #eee; padding: 6px; text-align: left; }
  tr.new td { background: #e6f6ec; }
  tr.changed td { background: #fff6d6; }
  pre { background: #f4f4f4; padding: 8px; white-space: pre-wrap; font-size: 12px; margin: 4px 0 0; }
  .log-item { border-bottom: 1px solid #eee; padding: 8px 0; font-size: 13px; }
  .badge { display: inline-block; padding: 1px 6px; border-radius: 4px; font-size: 12px; font-weight: 600; margin-right: 6px; }
  .GET { background: #e3effd; } .POST { background: #e6f6ec; } .PUT { background: #fff6d6; } .DELETE { background: #fde7e7; }
  .ok { color: #1a7f37; } .err { color: #c62828; } .muted { color: #888; }
  @media (max-width: 800px) { .grid { grid-template-columns: 1fr; } }
</style>
</head>
<body>
<h1>Winners API: live view</h1>
<p class="sub">The table and the request log refresh every second. Send requests with the buttons,
or run the test cells in the notebook and watch them appear here.</p>

<div class="grid">
  <div class="card">
    <h2>Send a request</h2>
    <label>API version
      <select id="version">
        <option value="/api/winners/">JSON + Marshmallow (/api/winners/)</option>
        <option value="/winners/">Text, no JSON (/winners/)</option>
      </select>
    </label>
    <input id="wid" placeholder="id (for GET one, PUT, DELETE)">
    <div class="row">
      <input id="name" placeholder="name">
      <input id="category" placeholder="category">
    </div>
    <input id="country" placeholder="country">
    <div>
      <button onclick="send('GET', false, false)">GET all</button>
      <button onclick="send('GET', true, false)">GET one</button>
      <button onclick="send('POST', false, true)">POST</button>
      <button onclick="send('PUT', true, true)">PUT</button>
      <button onclick="send('DELETE', true, false)">DELETE</button>
    </div>
    <h2 style="margin-top:14px">Last reply</h2>
    <pre id="reply">Nothing sent yet</pre>
  </div>

  <div class="card">
    <h2>Database table (live)</h2>
    <table id="tbl"></table>
    <p class="muted" style="font-size:12px">Green = new row, yellow = changed row (highlighted for 3 seconds)</p>
  </div>
</div>

<div class="card" style="margin-top:16px">
  <h2>Request log (from this page and from the notebook)</h2>
  <div id="log">No requests yet</div>
</div>

<script>
  const val = id => document.getElementById(id).value.trim();
  const esc = s => String(s ?? "").replace(/[&<>]/g, c => ({"&": "&amp;", "<": "&lt;", ">": "&gt;"}[c]));

  // Only the fields that were typed in, so PUT changes just those fields
  function fields() {
    const b = {};
    for (const f of ["name", "category", "country"]) { if (val(f)) b[f] = val(f); }
    return b;
  }

  async function send(method, needsId, hasBody) {
    const base = val("version");
    if (needsId && !val("wid")) { document.getElementById("reply").textContent = "Type an id first"; return; }
    const url = base + (needsId ? val("wid") : "");
    const options = { method: method };
    if (hasBody) {
      if (base.startsWith("/api/")) {          // JSON version: send JSON
        options.headers = { "Content-Type": "application/json" };
        options.body = JSON.stringify(fields());
      } else {                                 // text version: send form fields
        options.body = new URLSearchParams(fields());
      }
    }
    const res = await fetch(url, options);
    const text = await res.text();
    document.getElementById("reply").textContent = method + " " + url + "  ->  " + res.status + "\n\n" + text;
    refreshTable();
    refreshLog();
  }

  let previous = {};
  const marks = {};

  async function refreshTable() {
    try {
      const res = await fetch("/api/winners/", { headers: { "X-Live-View": "poll" } });
      const rows = await res.json();
      const now = {};
      let html = "<tr><th>id</th><th>name</th><th>category</th><th>country</th></tr>";
      for (const w of rows) {
        const key = JSON.stringify(w);
        now[w.id] = key;
        if (Object.keys(previous).length) {
          if (!(w.id in previous)) marks[w.id] = { cls: "new", until: Date.now() + 3000 };
          else if (previous[w.id] !== key) marks[w.id] = { cls: "changed", until: Date.now() + 3000 };
        }
        const m = marks[w.id];
        const cls = m && m.until > Date.now() ? m.cls : "";
        html += `<tr class="${cls}"><td>${w.id}</td><td>${esc(w.name)}</td><td>${esc(w.category)}</td><td>${esc(w.country)}</td></tr>`;
      }
      previous = now;
      document.getElementById("tbl").innerHTML = html;
    } catch (e) { /* server not reachable: keep the last view */ }
  }

  async function refreshLog() {
    try {
      const items = await (await fetch("/api/log")).json();
      if (!items.length) return;
      document.getElementById("log").innerHTML = items.map(it => `
        <div class="log-item">
          <span class="badge ${it.method}">${it.method}</span><b>${esc(it.path)}</b>
          <span class="${it.status < 400 ? "ok" : "err"}">${it.status}</span>
          <span class="muted">${it.time} · from ${it.source}</span>
          ${it.sent ? `<pre>sent:  ${esc(it.sent)}</pre>` : ""}
          <pre>reply: ${esc(it.reply)}</pre>
        </div>`).join("");
    } catch (e) { }
  }

  refreshTable();
  refreshLog();
  setInterval(() => { refreshTable(); refreshLog(); }, 1000);
</script>
</body>
</html>
"""


@app.route("/ui")
def ui():
    return UI_PAGE

### All routes the API exposes (text and JSON)

In [27]:
#List every route the API exposes

for rule in app.url_map.iter_rules():
    if rule.endpoint != "static":
        methods = ", ".join(sorted(rule.methods - {"HEAD", "OPTIONS"}))
        print(f"{methods:7} {rule.rule:20} -> {rule.endpoint}()")

GET     /                    -> home()
GET     /winners/            -> get_winners()
GET     /winners/<int:id>    -> get_winner()
POST    /winners/            -> create_winner()
PUT     /winners/<int:id>    -> update_winner()
DELETE  /winners/<int:id>    -> delete_winner()
GET     /api/winners/        -> api_get_winners()
GET     /api/winners/<int:id> -> api_get_winner()
POST    /api/winners/        -> api_create_winner()
PUT     /api/winners/<int:id> -> api_update_winner()
GET     /api/winners/search  -> search_winners()
DELETE  /api/winners/<int:id> -> api_delete_winner()
GET     /api/log             -> api_log()
GET     /ui                  -> ui()


## Part 3c – Testing the API inside the notebook (no running server)

The routes above **are** the API. `app.test_client()` sends requests to them directly inside the notebook,
skipping only the network: no server, no port, no browser or Postman needed. Flask still matches each URL to
its route, the route function runs, and the database file really changes.

After each request, `show_part3()` reads the table **directly** with SQL (not through the API), so you can
compare what the API *said* it did with what is *actually stored*.

### Open the live view (optional, recommended)

Run the next cell, then open **http://127.0.0.1:5000/ui** in your browser and keep it next to the notebook.
* Run the test cells below **one at a time**: each request appears in the request log, and new or changed
  rows light up in the table.
* Or use the buttons on the page yourself, switching between the JSON and the text version of the API.

The server runs in the background, so the notebook stays usable. The test client and the server use the
same app and the same database file, so changes from either one show up in both places.

In [28]:
# Start the real server in the BACKGROUND, so the notebook stays usable while it runs
import threading
from werkzeug.serving import make_server

# If this cell was run before, stop the old server first (it would still be serving the old app)
if globals().get("server") is not None:
    server.shutdown()
    server.server_close()

server = make_server("127.0.0.1", 5000, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()

print("Server running. Open the live view in your browser:  http://127.0.0.1:5000/ui")

Server running. Open the live view in your browser:  http://127.0.0.1:5000/ui


### **FOR STUDENTS READING**

### Text version (no serialization)

In [29]:
# Text routes: requests send form fields (data=), replies are read as text
# show_part3() is defined right after the seed cell

client = app.test_client()
show_part3("Before")

# READ ALL 
r = client.get("/winners/")
print("\nGET /winners/ ->", r.status_code)
print(r.get_data(as_text=True))

# CREATE 
r = client.post("/winners/", data={"name": "C.V. Raman", "category": "Physics", "country": "India"})
reply = r.get_data(as_text=True)
print("\nPOST ->", r.status_code, reply)

# The reply is only text, so the new id is cut out of the string by hand
new_id = int(reply.split("id=")[1].split()[0])
show_part3("After POST")

# UPDATE (only the field we send changes)
r = client.put(f"/winners/{new_id}", data={"country": "Independent India"})
print("\nPUT ->", r.status_code, r.get_data(as_text=True))
show_part3("After PUT")

# DELETE
r = client.delete(f"/winners/{new_id}")
print("\nDELETE ->", r.status_code, r.get_data(as_text=True))
show_part3("After DELETE")


 Before 
 id            name   category country
  1 Albert Einstein    Physics Germany
  2     Marie Curie  Chemistry  Poland
  3       Bob Dylan Literature     USA

GET /winners/ -> 200
<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'>

POST -> 201 <Winner id=4 name='C.V. Raman' category='Physics' country='India'>

 After POST 
 id            name   category country
  1 Albert Einstein    Physics Germany
  2     Marie Curie  Chemistry  Poland
  3       Bob Dylan Literature     USA
  4      C.V. Raman    Physics   India

PUT -> 200 <Winner id=4 name='C.V. Raman' category='Physics' country='Independent India'>

 After PUT 
 id            name   category           country
  1 Albert Einstein    Physics           Germany
  2     Marie Curie  Chemistry            Poland
  3       Bob Dylan Literature               USA
  4      C.

In [30]:
# same as above cell - just a simpler way
client = app.test_client()

def show(r):
    print(r.status_code, r.get_data(as_text=True), "\n")

show(client.get("/winners/"))      # GET all
show(client.get("/winners/1"))     # GET one (id 1)

show(client.post("/winners/", data={"name": "C.V. Raman", "category": "Physics", "country": "India"}))   # POST

show(client.put("/winners/4", data={"country": "Independent India"}))   # PUT (id 4)

show(client.delete("/winners/4"))  # DELETE (id 4)

show(client.get("/winners/"))      # GET all again, to check



200 <Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'> 

200 <Winner id=1 name='Albert Einstein' category='Physics' country='Germany'> 

201 <Winner id=4 name='C.V. Raman' category='Physics' country='India'> 

200 <Winner id=4 name='C.V. Raman' category='Physics' country='Independent India'> 

200 Deleted: <Winner id=4 name='C.V. Raman' category='Physics' country='Independent India'> 

200 <Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
<Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>
<Winner id=3 name='Bob Dylan' category='Literature' country='USA'> 



### JSON version (with Marshmallow)

The same requests, with two changes: `json=` instead of `data=`, and `r.get_json()` instead of
`r.get_data(as_text=True)`. The new id is a simple key lookup instead of cutting it out of text.

In [31]:
# test the JSON routes (put after flasks existing text-version test cells)

client = app.test_client()

def show_json(r):
    print(r.status_code, r.get_json(), "\n")

show_json(client.get("/api/winners/"))      # GET all
show_json(client.get("/api/winners/1"))     # GET one

r = client.post("/api/winners/", json={"name": "C.V. Raman", "category": "Physics", "country": "India"})   # POST
show_json(r)
new_id = r.get_json()["id"]                 # JSON: the id is a key lookup, no cutting text
show_part3("After POST")

show_json(client.put(f"/api/winners/{new_id}", json={"country": "Independent India"}))   # PUT
show_part3("After PUT")

show_json(client.delete(f"/api/winners/{new_id}"))   # DELETE
show_part3("After DELETE")



200 [{'category': 'Physics', 'country': 'Germany', 'id': 1, 'name': 'Albert Einstein'}, {'category': 'Chemistry', 'country': 'Poland', 'id': 2, 'name': 'Marie Curie'}, {'category': 'Literature', 'country': 'USA', 'id': 3, 'name': 'Bob Dylan'}] 

200 {'category': 'Physics', 'country': 'Germany', 'id': 1, 'name': 'Albert Einstein'} 

201 {'category': 'Physics', 'country': 'India', 'id': 4, 'name': 'C.V. Raman'} 


 After POST 
 id            name   category country
  1 Albert Einstein    Physics Germany
  2     Marie Curie  Chemistry  Poland
  3       Bob Dylan Literature     USA
  4      C.V. Raman    Physics   India
200 {'category': 'Physics', 'country': 'Independent India', 'id': 4, 'name': 'C.V. Raman'} 


 After PUT 
 id            name   category           country
  1 Albert Einstein    Physics           Germany
  2     Marie Curie  Chemistry            Poland
  3       Bob Dylan Literature               USA
  4      C.V. Raman    Physics Independent India
200 {'deleted': {'categor

### Text vs JSON, side by side

The same requests sent to both versions. The JSON version can be used directly by a program, and
Marshmallow's `load()` rejects data the text version accepts (form fields are always text, so a
number sent as `name` is not detected there).

In [32]:
#text vs JSON, side by side
# Same request, both versions, side by side
client = app.test_client()

r_text = client.get("/winners/1")
r_json = client.get("/api/winners/1")
print("GET one")
print("  text:", r_text.content_type, "|", r_text.get_data(as_text=True))
print("  json:", r_json.content_type, "|", r_json.get_json())

print("\nPOST with missing fields")
print("  text:", client.post("/winners/", data={"name": "Only a name"}).get_data(as_text=True))
print("  json:", client.post("/api/winners/", json={"name": "Only a name"}).get_json())

print("\nPOST with a wrong type (name is a number)")
r_text = client.post("/winners/", data={"name": 123, "category": "Physics", "country": "India"})
r_json = client.post("/api/winners/", json={"name": 123, "category": "Physics", "country": "India"})
print("  text:", r_text.status_code, r_text.get_data(as_text=True), " <- accepted: form fields are always text")
print("  json:", r_json.status_code, r_json.get_json())
bad_id = r_text.get_data(as_text=True).split("id=")[1].split()[0]
client.delete(f"/winners/{bad_id}")   # remove the bad row so the table stays clean
show_part3("After clean-up")

GET one
  text: text/plain; charset=utf-8 | <Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>
  json: application/json | {'category': 'Physics', 'country': 'Germany', 'id': 1, 'name': 'Albert Einstein'}

POST with missing fields
  text: Missing fields: category, country
  json: {'errors': {'category': ['Missing data for required field.'], 'country': ['Missing data for required field.']}}

POST with a wrong type (name is a number)
  text: 201 <Winner id=4 name='123' category='Physics' country='India'>  <- accepted: form fields are always text
  json: 400 {'errors': {'name': ['Not a valid string.']}}

 After clean-up 
 id            name   category country
  1 Albert Einstein    Physics Germany
  2     Marie Curie  Chemistry  Poland
  3       Bob Dylan Literature     USA


### Why `with app.app_context()` is needed

Outside a request, Flask-SQLAlchemy does not know which app (and so which database) to use.
This cell uses the database **without** an app context, so it fails on purpose; the error is caught
and printed so that Run All continues.

In [33]:
# Test why app.app_context() is needed in Flask
try:
    Winner.query.all()                      # no app context -> fails
except RuntimeError as err:
    print("Without app context ->", type(err).__name__, ":", str(err).splitlines()[0])

with app.app_context():
    print("With app context    ->", Winner.query.all())

Without app context -> RuntimeError : Working outside of application context.
With app context    -> [<Winner id=1 name='Albert Einstein' category='Physics' country='Germany'>, <Winner id=2 name='Marie Curie' category='Chemistry' country='Poland'>, <Winner id=3 name='Bob Dylan' category='Literature' country='USA'>]


**Explanation:** eg: Part 2 used plain SQLAlchemy, where we passed everything around explicitly: The session was bound to the engine directly, so there was never a question of which database to use, and no Flask was involved

### Stopping the server

The background server keeps running until you stop it or restart the kernel. If you change a route,
re-run from the app cell downwards: the start-server cell stops the old server and starts a new one
for the new app.

In [34]:
# Stop the background server (set to True and run this cell when you are done)
STOP_SERVER = True

if STOP_SERVER and globals().get("server") is not None:
    server.shutdown()
    server.server_close()
    server = None
    print("Server stopped")
else:
    print("Server still running (set STOP_SERVER = True to stop it). Restarting the kernel also stops it.")

Server stopped
